In [ ]:
from pathlib import Path
import shutil
import zipfile

content = Path('/content')
archives = sorted(content.glob('gemma_e2b_final_bundle*.zip'))
if not archives:
    raise FileNotFoundError(
        'Upload gemma_e2b_final_bundle.zip to /content before running the notebook.'
    )
archive = max(archives, key=lambda path: path.stat().st_mtime)
bundle_root = content / 'gemma_e2b_final_bundle'
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True)
with zipfile.ZipFile(archive) as handle:
    handle.extractall(bundle_root)
print(f'Extracted {archive.name} to {bundle_root}')
print((bundle_root / 'README.md').read_text(encoding='utf-8'))


In [ ]:
%pip install -q -r /content/gemma_e2b_final_bundle/requirements-colab.txt


In [ ]:
import subprocess
import torch

print(subprocess.check_output(['nvidia-smi'], text=True))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select an A100 GPU runtime and reconnect.')
gpu_name = torch.cuda.get_device_name(0)
if 'A100' not in gpu_name.upper():
    raise RuntimeError(f'The frozen final run expects an A100, but Colab reports {gpu_name!r}.')
print(f'Using {gpu_name}')


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    os.environ['HF_TOKEN'] = hf_token
    print('Authenticated to Hugging Face using the HF_TOKEN Colab secret.')
else:
    from huggingface_hub import notebook_login
    print('HF_TOKEN was not found; complete the interactive Hugging Face login below.')
    notebook_login()


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

bundle_root = Path('/content/gemma_e2b_final_bundle')
local_root = Path('/content/gemma_e2b_final_work')
drive_root = Path(
    '/content/drive/MyDrive/Dissertation_Gemma_Final/'
    'gemma_e2b_counterfactual_monitorability_v1'
)

command = [
    sys.executable,
    str(bundle_root / 'src' / 'run_final_gemma_study.py'),
    '--bundle-root', str(bundle_root),
    '--local-root', str(local_root),
    '--drive-root', str(drive_root),
    '--stages', 'all',
]
environment = os.environ.copy()
subprocess.run(command, check=True, env=environment)


In [ ]:
from pathlib import Path

drive_root = Path(
    '/content/drive/MyDrive/Dissertation_Gemma_Final/'
    'gemma_e2b_counterfactual_monitorability_v1'
)
completion = drive_root / 'RUN_COMPLETE.json'
quality = drive_root / 'QUALITY_GATE.json'
summary = drive_root / 'analysis' / 'automatic_summary.md'
print(completion.read_text(encoding='utf-8'))
print('\nFinal structural quality gate:')
print(quality.read_text(encoding='utf-8'))
print('\n' + summary.read_text(encoding='utf-8'))
print('\nResult archives:')
for path in sorted(drive_root.parent.glob(drive_root.name + '_results*.zip')):
    print(path)
